# Lab 07: 循環神經網路 (RNN) 實作 — 自動英文字元生成 (使用真實嬰兒姓名資料集)

在本單元中，我們將跨入 **自然語言處理 (NLP)** 的領域。
我們將學習什麼是序列資料，並實作一個 **字元級 (Character-level) RNN 模型**。我們將從網路上下載含有 32,000 多筆資料的 **真實美國嬰兒姓名資料集 (US Baby Names Dataset)**，訓練它學會字母的排序邏輯，最後輸入一個起頭字母，讓它幫我們「創造」出全新的名字！

### 任務 1: 從網路載入真實嬰兒姓名資料集
我們使用 Python 內建的 `urllib` 從公開 GitHub 載入真實的姓名資料庫。
*(貼心設計：若電腦教室因為網路限制下載失敗，會自動切換為本地預設姓名集，確保教學不受影響！)*

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
import urllib.request

# 預設本地備用名單
local_backup_names = [
    "Alice", "Amber", "Andrew", "Andy", "Anna", "Austin", "Benjamin", "Bruce",
    "Carol", "Catherine", "Charles", "Chris", "Daniel", "David", "Diana",
    "Edward", "Emily", "Eric", "Fiona", "Frank", "George", "Grace",
    "Henry", "Harry", "Ian", "Jack", "Jane", "Jason", "John", "Kevin",
    "Leo", "Lily", "Lucy", "Mark", "Mary", "Nancy", "Oliver", "Oscar",
    "Peter", "Patrick", "Ryan", "Rose", "Sarah", "Steven", "Tom", "Tony",
    "Victor", "Vincent", "Wendy", "William", "Zoe"
]

names_url = "https://raw.githubusercontent.com/karpathy/makemore/master/names.txt"

try:
    print("正在從網路下載真實美國嬰兒姓名資料集 (32,000+筆)...")
    headers = {'User-Agent': 'Mozilla/5.0'}
    req = urllib.request.Request(names_url, headers=headers)
    with urllib.request.urlopen(req, timeout=5) as response:
        words = response.read().decode('utf-8').splitlines()
    # 首字母大寫，並選取前 5000 個名字進行快速訓練
    names = [w.capitalize() for w in words[:5000] if w.isalpha()]
    print(f"下載完成！共讀入 {len(words)} 個姓名，已選取前 {len(names)} 個進行訓練。")
except Exception as e:
    print(f"網路下載失敗 ({e})。啟動本地備用姓名集...")
    names = local_backup_names

# 建立字元表與對照 dictionary
all_chars = sorted(list(set("".join(names))))
all_chars.append('<EOS>') # 結尾符號

char_to_idx = {char: idx for idx, char in enumerate(all_chars)}
idx_to_char = {idx: char for idx, char in enumerate(all_chars)}
vocab_size = len(all_chars)

print(f"字元庫大小 (Vocab Size): {vocab_size}")
print("前五個名字範例:", names[:5])

### 任務 2: 輔助函數 — 單字與張量轉換
我們需要將一個名字轉換為張量。例如 "Andy"：
- 輸入序列為： `['A', 'n', 'd', 'y']` 轉為對應數字。
- 預期的正確目標序列為： `['n', 'd', 'y', '<EOS>']` 轉為對應數字（預測下一個字元）。

In [ ]:
def word_to_tensors(word):
    # 輸入序列 (不含最後一個 <EOS>)
    input_idxs = [char_to_idx[c] for c in word]
    # 目標序列 (往前移一位，最後加上 <EOS>)
    target_idxs = [char_to_idx[c] for c in word[1:]] + [char_to_idx['<EOS>']]
    
    # 形狀為 [1, seq_len]，增加 Batch 維度
    input_tensor = torch.tensor(input_idxs, dtype=torch.long).unsqueeze(0)
    target_tensor = torch.tensor(target_idxs, dtype=torch.long).unsqueeze(0)
    return input_tensor, target_tensor

x_test, y_test = word_to_tensors("Andy")
print("Andy 輸入張量:", x_test)
print("Andy 目標張量:", y_test)

### 任務 3: 定義字元級 RNN 生成器
我們建構包含嵌入層 字詞空間化 (Embedding) `nn.Embedding`（把字母轉換成向量）、循環層 `nn.RNN` 與線性層的結構。

In [ ]:
class RNNGenerator(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super(RNNGenerator, self).__init__()
        self.hidden_size = hidden_size
        
        # 1. 嵌入層: 將字元索引轉為特徵向量
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        
        # 2. TODO: 定義一個 nn.RNN 層，輸入與輸出特徵數皆為 hidden_size，且 batch_first=True
        # 提示：self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        # ??? (請在此填寫你的答案)
        
        # 3. 輸出層: 預測下一個最可能是哪個字元 (輸出長度為字元表大小)
        self.fc = nn.Linear(hidden_size, vocab_size)
        
    def forward(self, x, hidden):
        # x 形狀: [1, seq_len]
        x = self.embedding(x) # 轉為 [1, seq_len, hidden_size]
        out, hidden = self.rnn(x, hidden)
        out = self.fc(out)
        return out, hidden

hidden_dim = 64
model = RNNGenerator(vocab_size, hidden_dim)
print(model)

### 任務 4: 訓練 RNN 模型
我們進行 5000 次隨機單字隨機梯度下降訓練。

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.002)

print("開始訓練文字生成 RNN 模型...")
for step in range(5000):
    # 隨機挑選一個真實名字進行訓練
    word = random.choice(names)
    input_tensor, target_tensor = word_to_tensors(word)
    
    # 初始化 RNN 的記憶 (Hidden State) 為全零，形狀: [層數, Batch, Hidden大小] = [1, 1, 64]
    hidden = torch.zeros(1, 1, hidden_dim)
    
    # 模型預測計算 (前向傳播)
    outputs, hidden = model(input_tensor, hidden)
    
    # 重塑形狀以符合 CrossEntropy 期望: [seq_len, vocab_size] vs [seq_len]
    loss = criterion(outputs.view(-1, vocab_size), target_tensor.view(-1))
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if (step + 1) % 1000 == 0:
        print(f"Step [{step+1}/5000], Loss: {loss.item():.4f}")

### 任務 5: 自動英文字母名字生成！
我們寫一個推理函數，輸入第一個英文字母，讓模型一個字一個字預測，直到產生 `<EOS>` 結束生成。

In [ ]:
def generate_name(start_letter, max_len=15, temperature=0.8):
    model.eval()
    with torch.no_grad():
        # 驗證起頭字母是否在我們的字元表中
        if start_letter not in char_to_idx:
            return "此字母無法起頭，請輸入有大寫英文開頭的字！"
            
        # 建立起頭字母張量
        curr_char_idx = char_to_idx[start_letter]
        input_tensor = torch.tensor([[curr_char_idx]], dtype=torch.long)
        
        generated_name = start_letter
        hidden = torch.zeros(1, 1, hidden_dim)
        
        for _ in range(max_len):
            # (已幫你完成) 讓模型進行預測計算，取得輸出 outputs 與新的記憶 h
            outputs, hidden = model(input_tensor, hidden)
            
            # 透過 Temperature 與 Softmax 取得下一個字元的機率分佈 (解決原 max(0) 每次生成都一樣的問題)
            logits = outputs[0, -1] / temperature
            probs = torch.softmax(logits, dim=0)
            
            # 依據機率分佈進行多項式隨機抽樣，以產生多樣且隨機的名字
            next_idx = torch.multinomial(probs, num_samples=1).item()
            
            # 如果是 <EOS>，代表單字生成結束
            if idx_to_char[next_idx] == '<EOS>':
                break
                
            # 將預測的字元拼接到名字中
            generated_name += idx_to_char[next_idx]
            
            # 將這個預測的字元作為下一次的輸入
            input_tensor = torch.tensor([[next_idx]], dtype=torch.long)
            
        return generated_name

# 測試隨機生成結果：同一個起頭字母多次生成不同的名字！
print("--- 隨機生成展示：同一個起頭字母多樣輸出 ---")
for _ in range(3):
    print(f"起頭字母: A → 生成英文名: {generate_name('A', temperature=0.8)}")

print("\n--- 不同字母的隨機名字生成 ---")
for start in ['A', 'B', 'L', 'S', 'T']:
    new_name = generate_name(start, temperature=0.8)
    print(f"起頭字母: {start} → 生成英文名: {new_name}")

---

## Lab 07 學習重點小結
### 核心概念掌握
- RNN (循環神經網路) 透過隱藏狀態 h_t 將前一時間步的資訊傳遞到下一步。
- 自回歸接龍：模型預測下一個字元，並將預測結果作為下一步的輸入。
- 字元級 RNN 以單個字元為單位，可生成任意長度的文字序列。
- RNN 的弱點：梯度消失問題使其難以學習長期依賴關係。

### 快速自評測驗

**Q1. RNN 中，隱藏狀態 h_t 的來源為？**
- A. 僅來自當前輸入 x_t
- B. 來自 x_t 與前一隱藏狀態 h_(t-1) 的結合
- C. 來自輸出層的回饋

<details><summary>查看解答</summary>

**答案：B — h_t = tanh(W_h * h_(t-1) + W_x * x_t + b)，同時考慮當前輸入與記憶。**

</details>

**Q2. 自回歸生成的意思是？**
- A. 模型同時預測所有字元
- B. 模型依序預測，並以自己的預測結果作為下一步輸入
- C. 模型僅輸出固定長度序列

<details><summary>查看解答</summary>

**答案：B — 每一步的輸出回饋為下一步的輸入，形成序列生成鏈。**

</details>

### 完成確認清單
在繼續下一個 Lab 前，請確認你能做到：
- [ ] 我可以用一句話解釋「RNN (循環神經網路) 透過隱藏狀態 h_t 將前一時間步」
- [ ] 我可以用一句話解釋「自回歸接龍」
- [ ] 我可以用一句話解釋「字元級 RNN 以單個字元為單位，可生成任意長度的文字序列。」
- [ ] 我的程式碼成功執行並得到預期輸出
